# 01. Data Audit and Data Quality Assessment

**Notebook objective:** audit the source CRM and marketing datasets against the rules defined in the [Data Dictionary](../docs/data_dictionary.pdf).

At this stage, the data is not cleaned yet. The goal is to identify data quality issues, validate table relationships, and define what should be handled during the cleaning stage.


In [40]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import sys

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)


# Loading Source Data

The source files are stored in the `data/raw/` folder of the project repository.

The datasets are anonymized and do not contain customer names, emails or phone numbers. Technical CRM-style identifiers are preserved because they are required for relationship checks between tables. Synthetic manager names are used only for analytical grouping.

At this stage, I work only with copies of the source tables and do not modify the original data.


In [41]:
# Define project paths so the notebook can be run from the repository structure

current_path = Path.cwd()

# If the notebook is opened from the notebooks folder, the project root is one level above
PROJECT_ROOT = current_path.parent if current_path.name == 'notebooks' else current_path

RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw'

RAW_DATA_PATH


PosixPath('/Users/uliakardenasvalin/Documents/Аналитик данных/online-school-performance-analytics-portfolio/data/raw')

In [42]:
# Add the notebooks folder to the Python path for custom helper functions

NOTEBOOKS_PATH = PROJECT_ROOT / 'notebooks'
sys.path.append(str(NOTEBOOKS_PATH))

from project_util import (
    clean_id_column,
    clean_text_column,
    clean_money_column,
    dict_to_check_df
)


In [43]:
# Check that the raw data folder is available

sorted([file.name for file in RAW_DATA_PATH.iterdir()])


['calls_raw.xlsx', 'contacts_raw.xlsx', 'deals_raw.xlsx', 'spend_raw.xlsx']

### Loading CRM Identifiers

CRM tables use long technical identifiers. These fields are not numerical metrics and should not be treated as numbers.

If long IDs are read as `float`, the last digits may be distorted due to precision loss. This can lead to incorrect joins between tables.

For this reason, `Id`, `CONTACTID` and `Contact Name` are loaded and processed as strings.

In [44]:
spend = pd.read_excel(RAW_DATA_PATH / 'spend_raw.xlsx')

contacts = pd.read_excel(
    RAW_DATA_PATH / 'contacts_raw.xlsx',
    dtype={'Id': 'string'}
)

calls = pd.read_excel(
    RAW_DATA_PATH / 'calls_raw.xlsx',
    dtype={
        'Id': 'string',
        'CONTACTID': 'string'
    }
)

deals = pd.read_excel(
    RAW_DATA_PATH / 'deals_raw.xlsx',
    dtype={
        'Id': 'string',
        'Contact Name': 'string'
    }
)


# Initial Dataset Overview

At the first stage, I review the overall structure and volume of the data.

The checks include:
- dataset size;
- first rows of each table;
- available columns and general data structure.

This helps determine:
- which tables are central for the analysis;
- whether the data is suitable for further analysis;
- how different project tables can be connected.

In [45]:
datasets = {
    'Deals': deals,
    'Calls': calls,
    'Contacts': contacts,
    'Spend': spend
}

summary = []

for name, df in datasets.items():
    summary.append({
        'Dataset': name,
        'Records': df.shape[0],
        'Columns': df.shape[1],
        'Full duplicates': df.duplicated().sum(),
        'Total missing values': df.isna().sum().sum(),
        'Columns with missing values': (df.isna().sum() > 0).sum()
    })

summary_df = pd.DataFrame(summary)
summary_df

,Dataset,Records,Columns,Full duplicates,Total missing values,Columns with missing values
0,Deals,21595,23,0,213387,23
1,Calls,95874,11,0,213762,6
2,Contacts,18548,4,0,0,0
3,Spend,20779,8,917,19650,3


## Initial Dataset Overview: Key Findings

The project includes 4 tables: `Deals`, `Calls`, `Contacts` and `Spend`.

The dataset contains enough data for business analysis and insight generation:

- the main table `Deals` contains 21,595 records;
- the `Calls` table contains 95,874 call records;
- the `Contacts` table contains 18,548 contacts;
- the `Spend` table contains 20,779 marketing spend records.

The main table for the analysis is `Deals`, because it contains information about deals, stages, sources, products and payments.

The `Spend` and `Calls` tables are used for additional analysis of marketing spend and sales activity. `Contacts` is used as a CRM contact reference table.

---

The initial overview also shows that data quality differs across tables:

- `Contacts` looks the most complete: no missing values or full duplicates were found;
- `Deals` contains missing values in all 23 columns and therefore requires the most careful validation and cleaning;
- `Calls` contains missing values in 6 columns, and some fields may not be informative for the analysis;
- `Spend` contains 917 full duplicates that need to be checked before marketing spend metrics are calculated.

The next step is to run a detailed data quality audit for each table based on the rules described in the Data Dictionary.

# Data Quality Audit: `Spend`

The `Spend` table contains data on marketing spend, impressions and clicks.  
I check the table against the rules defined in the Data Dictionary:

- missing values in key fields;
- full duplicates;
- negative values in numerical fields;
- logical errors in the relationship between clicks and impressions;
- correctness of the marketing source list.

In [46]:
# List of allowed lead acquisition sources
allowed_sources = [
    'Bloggers',
    'CRM',
    'Facebook Ads',
    'Google Ads',
    'Offline',
    'Organic',
    'Partnership',
    'Radio',
    'SMM',
    'Telegram posts',
    'Test',
    'Tiktok Ads',
    'Webinar',
    'Youtube Ads'
]

spend_check = spend.copy()

# Check whether key fields can be converted to expected data types
spend_check['Date_check'] = pd.to_datetime(spend_check['Date'], errors='coerce')

for col in ['Impressions', 'Spend', 'Clicks']:
    spend_check[col + '_check'] = pd.to_numeric(spend_check[col], errors='coerce')


# Data quality check table
spend_checks = {
    'Records': spend.shape[0],
    'Columns': spend.shape[1],
    'Full duplicates': spend.duplicated().sum(),

    'Missing Date': spend_check['Date'].isna().sum(),
    'Date conversion errors': spend_check['Date_check'].isna().sum(),

    'Minimum date': spend_check['Date_check'].min(),
    'Maximum date': spend_check['Date_check'].max(),

    'Missing Source': spend_check['Source'].isna().sum(),
    'Source values outside the allowed list': (~spend_check['Source'].isin(allowed_sources)).sum(),

    'Missing Campaign': spend_check['Campaign'].isna().sum(),
    'Missing Campaign where Spend > 0': (
        spend_check['Campaign'].isna() & (spend_check['Spend_check'] > 0)
    ).sum(),

    'Missing Impressions': spend_check['Impressions'].isna().sum(),
    'Impressions conversion errors': spend_check['Impressions_check'].isna().sum(),
    'Negative Impressions values': (spend_check['Impressions_check'] < 0).sum(),

    'Missing Spend': spend_check['Spend'].isna().sum(),
    'Spend conversion errors': spend_check['Spend_check'].isna().sum(),
    'Negative Spend values': (spend_check['Spend_check'] < 0).sum(),

    'Missing Clicks': spend_check['Clicks'].isna().sum(),
    'Clicks conversion errors': spend_check['Clicks_check'].isna().sum(),
    'Negative Clicks values': (spend_check['Clicks_check'] < 0).sum(),

    'Rows where Clicks exceed Impressions': (
        spend_check['Clicks_check'] > spend_check['Impressions_check']
    ).sum(),

    'Missing AdGroup': spend_check['AdGroup'].isna().sum(),
    'Missing AdGroup where Ad is filled': (
        spend_check['AdGroup'].isna() & spend_check['Ad'].notna()
    ).sum(),

    'Missing Ad': spend_check['Ad'].isna().sum()
}

spend_checks_df = pd.DataFrame(
    spend_checks.items(),
    columns=['Check', 'Result']
)

spend_checks_df

,Check,Result
0,Records,20779
1,Columns,8
2,Full duplicates,917
3,Missing Date,0
4,Date conversion errors,0
5,Minimum date,2023-07-03 00:00:00
6,Maximum date,2024-06-21 00:00:00
7,Missing Source,0
8,Source values outside the allowed list,0
9,Missing Campaign,5994


### Detailed Analysis of Full Duplicates in `Spend`

The `Spend` table contains 917 full duplicate rows. Before cleaning, I check examples of these rows and assess whether they are technical export duplicates.

In [47]:
spend_duplicates = spend_check[
    spend_check.duplicated(keep=False)
].sort_values(
    by=['Date', 'Source', 'Campaign', 'AdGroup', 'Ad']
)

spend_duplicates[['Date', 'Source', 'Campaign', 'Impressions', 'Spend', 'Clicks', 'AdGroup', 'Ad']]

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
753,2023-07-23,Bloggers,NaN,0,0.00,0,NaN,NaN
755,2023-07-23,Bloggers,NaN,0,0.00,0,NaN,NaN
768,2023-07-24,Bloggers,NaN,0,0.00,0,NaN,NaN
789,2023-07-24,Bloggers,NaN,0,0.00,0,NaN,NaN
841,2023-07-25,Bloggers,NaN,0,0.00,0,NaN,NaN
...,...,...,...,...,...,...,...,...
20773,2024-06-21,Organic,NaN,0,0.00,0,NaN,NaN
20738,2024-06-21,SMM,NaN,0,0.00,0,NaN,NaN
20750,2024-06-21,SMM,NaN,0,0.00,0,NaN,NaN
20729,2024-06-21,Telegram posts,NaN,0,0.00,0,NaN,NaN


In [48]:
duplicate_metrics_mean = pd.DataFrame({
    'Average across full Spend table': spend_check[
        ['Impressions_check', 'Spend_check', 'Clicks_check']
    ].mean(),
    'Average across full duplicates': spend_duplicates[
        ['Impressions_check', 'Spend_check', 'Clicks_check']
    ].mean()
})

duplicate_metrics_mean.index = ['Impressions', 'Spend', 'Clicks']

duplicate_metrics_mean

,Average across full Spend table,Average across full duplicates
Impressions,"2,458.20",0.00
Spend,7.20,0.00
Clicks,23.99,0.06


In [49]:
print('Number of rows in Spend:', spend_check.shape[0])
print('Number of full duplicates:', spend_check.duplicated().sum())
print('Number of rows after removing full duplicates:', spend_check.drop_duplicates().shape[0])


Number of rows in Spend: 20779
Number of full duplicates: 917
Number of rows after removing full duplicates: 19862


### Full Duplicates in `Spend`: Key Findings

The `Spend` table contains 917 full duplicates.

Additional checks showed that the average values of key numerical metrics for duplicated rows are close to zero:

- average `Impressions` for duplicates is 0;
- average `Spend` for duplicates is 0;
- average `Clicks` for duplicates is close to 0.

This suggests that these rows are most likely related to no advertising activity or incomplete data granularity rather than duplicated meaningful ad spend.

From the perspective of key marketing metrics (`Spend`, `Impressions`, `Clicks`), these duplicates do not materially affect the results. Therefore, at this stage they are not treated as a critical data issue.

For further analysis, this should still be documented: duplicates may affect row counts and distribution of records by source, but they do not distort the key totals for spend, impressions and clicks.

### Additional Check: Rows Where `Clicks` Exceed `Impressions`

I check rows where the number of clicks is greater than the number of impressions.  
Such rows are logically incorrect for CTR calculation, so it is important to assess how significant they are across key metrics: `Impressions`, `Spend` and `Clicks`.

In [50]:
clicks_more_than_impressions = spend_check[
    spend_check['Clicks_check'] > spend_check['Impressions_check']
]

clicks_more_than_impressions[['Date', 'Source', 'Campaign', 'Impressions', 'Clicks', 'Spend', 'AdGroup', 'Ad']]

,Date,Source,Campaign,Impressions,Clicks,Spend,AdGroup,Ad
13,2023-07-03,SMM,NaN,0,5,0.00,NaN,NaN
15,2023-07-03,Organic,NaN,0,48,0.00,NaN,NaN
17,2023-07-04,Organic,NaN,0,37,0.00,NaN,NaN
24,2023-07-04,SMM,NaN,0,2,0.00,NaN,NaN
38,2023-07-05,Organic,NaN,0,43,0.00,NaN,NaN
...,...,...,...,...,...,...,...,...
20649,2024-06-20,CRM,NaN,0,14,0.00,NaN,NaN
20667,2024-06-20,Organic,NaN,0,11,0.00,NaN,NaN
20675,2024-06-20,SMM,NaN,0,1,0.00,NaN,NaN
20681,2024-06-20,Partnership,NaN,0,1,0.00,NaN,NaN


In [51]:
clicks_more_than_impressions_mean = pd.DataFrame({
    'Average across full Spend table': spend_check[
        ['Impressions_check', 'Spend_check', 'Clicks_check']
    ].mean(),
    'Average for rows where Clicks > Impressions': clicks_more_than_impressions[
        ['Impressions_check', 'Spend_check', 'Clicks_check']
    ].mean()
})

clicks_more_than_impressions_mean.index = ['Impressions', 'Spend', 'Clicks']

clicks_more_than_impressions_mean

,Average across full Spend table,Average for rows where Clicks > Impressions
Impressions,"2,458.20",0.00
Spend,7.20,5.04
Clicks,23.99,57.70


### Rows Where `Clicks > Impressions`: Key Findings

The `Spend` table contains rows where the number of clicks is greater than the number of impressions.

Additional checks showed that these rows have an average `Impressions` value of 0, while `Spend` and `Clicks` contain non-zero values.

**This means that these rows should not be removed completely:** they still contain useful information about marketing spend and clicks. The issue is likely related to missing or incorrectly exported impression data.

In further analysis, these rows:
- can be used for spend, clicks and CPC analysis;
- should not be used for CTR calculation;
- should not be used for reach and impression efficiency analysis.

At the cleaning stage, a data quality flag should be created for such rows, for example `invalid_ctr = True`.

## `Spend` Data Quality Summary

Based on the audit of the `Spend` table, the main findings are:

- The table contains 20,779 records. The available analysis period is from 2023-07-03 to 2024-06-21.
- Key fields `Date`, `Source`, `Impressions`, `Spend` and `Clicks` have no missing values. According to the Data Dictionary, missing values are not allowed for these fields.
- `Date` can be correctly converted to date format; no conversion errors were found.
- `Impressions`, `Spend` and `Clicks` can be correctly converted to numerical format. No non-numeric or negative values were found in these fields.
- Values in `Source` match the list of allowed sources from the Data Dictionary. No additional source standardization is required at this stage.
- The table contains 917 full duplicates. A detailed check showed that average `Impressions` and `Spend` for these rows are 0, and average `Clicks` is close to 0. This means that the duplicates do not materially affect key marketing metrics, although they may affect row counts.
- `Campaign` contains 5,994 missing values. Among them, 394 rows have `Spend > 0`. These rows require additional handling because marketing spend exists but the campaign is not specified.
- `AdGroup` and `Ad` each contain 6,828 missing values. No rows were found where `AdGroup` is missing while `Ad` is filled. This may indicate that both fields are missing due to the absence of ad group/ad-level granularity.
- There are 1,370 rows where `Clicks` exceed `Impressions`. A detailed check showed that these rows have `Impressions = 0`, while `Spend` and `Clicks` are non-zero. These rows should not be removed completely, but they should be excluded from CTR and impression efficiency analysis.

Overall, the `Spend` table is suitable for further marketing spend analysis. The main issues to handle before metric calculation are full duplicates, missing `Campaign` values where spend exists, and rows with an invalid relationship between `Clicks` and `Impressions`.

### Preliminary Cleaning Strategy for `Spend`

At the cleaning stage, the following rules will be applied to the `Spend` table:

- Keep `Date`, `Source`, `Impressions`, `Spend` and `Clicks` as key fields for marketing analysis because they are filled and match the expected data types.
- Do not automatically remove full duplicates. Since they contain almost no advertising activity, they do not materially affect total `Spend`, `Impressions` and `Clicks`. However, their presence should be considered when analyzing row counts and record distribution by source.
- For rows where `Campaign` is missing and `Spend > 0`, create a technical campaign value using the `Date_Source` rule, so that marketing spend is not lost in campaign analysis.
- Leave missing `Campaign` values unchanged when `Spend = 0`, because these rows do not contain marketing spend.
- Leave missing `AdGroup` and `Ad` values unchanged, because they may indicate missing granularity rather than an error.
- For rows where `Clicks > Impressions`, create a separate data quality flag, for example `invalid_ctr = True`.
- Keep rows with `Clicks > Impressions` for spend, clicks, CPC and ad-to-lead/deal connection analysis.
- Exclude rows with `Clicks > Impressions` from CTR calculation and impression efficiency analysis, because the `Impressions` value is invalid for these rows.

# Data Quality Audit: `Contacts`

I check the `Contacts` table against the rules defined in the Data Dictionary:

- presence of a unique contact identifier;
- missing values in key fields;
- correctness of contact creation and modification dates;
- logical check: `Modified Time` should not be earlier than `Created Time`.

In [52]:
contacts_check = contacts.copy()

# Clean identifiers to correctly check keys and table relationships
contacts_check['Id_clean'] = clean_id_column(contacts_check['Id'])

# Check whether dates can be converted to the expected data type
contacts_check['Created Time_check'] = pd.to_datetime(
    contacts_check['Created Time'],
    errors='coerce',
    dayfirst=True
)

contacts_check['Modified Time_check'] = pd.to_datetime(
    contacts_check['Modified Time'],
    errors='coerce',
    dayfirst=True
)

contacts_checks = {
    'Records': contacts.shape[0],
    'Columns': contacts.shape[1],
    'Full duplicates': contacts.duplicated().sum(),

    'Missing Id': contacts_check['Id_clean'].isna().sum(),
    'Duplicate Id values': contacts_check['Id_clean'].dropna().duplicated().sum(),

    'Missing Contact Owner Name': contacts_check['Contact Owner Name'].isna().sum(),

    'Missing Created Time': contacts_check['Created Time'].isna().sum(),
    'Created Time conversion errors': contacts_check['Created Time_check'].isna().sum(),

    'Missing Modified Time': contacts_check['Modified Time'].isna().sum(),
    'Modified Time conversion errors': contacts_check['Modified Time_check'].isna().sum(),

    'Rows where Modified Time is earlier than Created Time': (
        contacts_check['Modified Time_check'] < contacts_check['Created Time_check']
    ).sum()
}

contacts_checks_df = pd.DataFrame(
    contacts_checks.items(),
    columns=['Check', 'Result']
)

contacts_checks_df

,Check,Result
0,Records,18548
1,Columns,4
2,Full duplicates,0
3,Missing Id,0
4,Duplicate Id values,0
5,Missing Contact Owner Name,0
6,Missing Created Time,0
7,Created Time conversion errors,0
8,Missing Modified Time,0
9,Modified Time conversion errors,0


In [53]:
contacts_dates_period = pd.DataFrame({
    'Date': ['Minimum Created Time', 'Maximum Created Time',
             'Minimum Modified Time', 'Maximum Modified Time'],
    'Value': [
        contacts_check['Created Time_check'].min(),
        contacts_check['Created Time_check'].max(),
        contacts_check['Modified Time_check'].min(),
        contacts_check['Modified Time_check'].max()
    ]
})

contacts_dates_period

,Date,Value
0,Minimum Created Time,2023-06-27 11:28:00
1,Maximum Created Time,2024-06-21 15:30:00
2,Minimum Modified Time,2023-07-06 10:54:00
3,Maximum Modified Time,2024-06-21 15:32:00


## `Contacts` Data Quality Summary

Based on the audit of the `Contacts` table, the main findings are:

- The table contains 18,548 records and 4 columns.
- No full duplicates were found.
- No missing values were found in `Id`, `Contact Owner Name`, `Created Time` or `Modified Time`.
- No duplicates were found in `Id`, so this field can be used as a unique contact identifier.
- `Created Time` and `Modified Time` can be correctly converted to datetime format.
- No rows were found where `Modified Time` is earlier than `Created Time`.

Overall, the `Contacts` table looks complete and well-structured. It can be used as a CRM contact reference table and linked to other tables through the contact identifier.

### Preliminary Cleaning Strategy for `Contacts`

At the cleaning stage, the following rules will be applied to the `Contacts` table:

- convert `Created Time` and `Modified Time` to datetime format;
- convert `Id` to string type, because it is a technical identifier rather than a numerical metric;
- keep `Contact Owner Name` unchanged unless different spellings of the same manager are found;
- do not remove rows, because no missing values, full duplicates or date logic errors were found.

# Data Quality Audit: `Calls`

I check the `Calls` table against the rules defined in the Data Dictionary:

- presence of a unique call identifier;
- presence and correctness of call date/time;
- completeness of manager, call type and call status fields;
- availability of a link to a contact through `CONTACTID`;
- correctness of call duration;
- fully empty fields that should not be used in the analysis.

In [54]:
calls_check = calls.copy()

# Clean identifiers to correctly check keys and table relationships
calls_check['Id_clean'] = clean_id_column(calls_check['Id'])
calls_check['CONTACTID_clean'] = clean_id_column(calls_check['CONTACTID'])

# Check whether call date can be converted to the expected data type
calls_check['Call Start Time_check'] = pd.to_datetime(
    calls_check['Call Start Time'],
    errors='coerce',
    dayfirst=True
)

# Check whether call duration can be converted to numerical type
calls_check['Call Duration_check'] = pd.to_numeric(
    calls_check['Call Duration (in seconds)'],
    errors='coerce'
)

calls_checks = {
    'Records': calls_check.shape[0],
    'Columns': calls.shape[1],
    'Full duplicates': calls_check.duplicated().sum(),

    'Missing Id': calls_check['Id_clean'].isna().sum(),
    'Duplicate Id values': calls_check['Id_clean'].dropna().duplicated().sum(),

    'Missing Call Start Time': calls_check['Call Start Time'].isna().sum(),
    'Call Start Time conversion errors': calls_check['Call Start Time_check'].isna().sum(),

    'Missing Call Owner Name': calls_check['Call Owner Name'].isna().sum(),

    'Missing CONTACTID': calls_check['CONTACTID_clean'].isna().sum(),

    'Missing Call Type': calls_check['Call Type'].isna().sum(),
    'Missing Call Status': calls_check['Call Status'].isna().sum(),

    'Missing Call Duration': calls_check['Call Duration (in seconds)'].isna().sum(),
    'Call Duration conversion errors': calls_check['Call Duration_check'].isna().sum(),
    'Negative Call Duration values': (calls_check['Call Duration_check'] < 0).sum(),

    'Missing Outgoing Call Status': calls_check['Outgoing Call Status'].isna().sum(),
    'Missing Scheduled in CRM': calls_check['Scheduled in CRM'].isna().sum(),

    'Missing Dialled Number': calls_check['Dialled Number'].isna().sum(),
    'Missing Tag': calls_check['Tag'].isna().sum()
}

calls_checks_df = pd.DataFrame(
    calls_checks.items(),
    columns=['Check', 'Result']
)

calls_checks_df

,Check,Result
0,Records,95874
1,Columns,11
2,Full duplicates,0
3,Missing Id,0
4,Duplicate Id values,0
5,Missing Call Start Time,0
6,Call Start Time conversion errors,0
7,Missing Call Owner Name,0
8,Missing CONTACTID,3933
9,Missing Call Type,0


### Checking the Relationship Between `Calls` and `Contacts`

I check how reliably the `Calls` table can be linked to the `Contacts` table through `CONTACTID`.

Cleaned string identifiers are used for this check, because CRM IDs are technical keys and should not be processed as numerical values.

In [55]:
contacts_ids = set(contacts_check['Id_clean'].dropna())

calls_with_contactid = calls_check[
    calls_check['CONTACTID_clean'].notna()
].copy()

unique_calls_contactids = calls_with_contactid['CONTACTID_clean'].drop_duplicates()

contact_match_check = {
    'Unique Id values in Contacts': contacts_check['Id_clean'].nunique(),
    'Unique CONTACTID values in Calls': unique_calls_contactids.nunique(),

    'Unique CONTACTID values from Calls found in Contacts': unique_calls_contactids.isin(contacts_ids).sum(),
    'Unique CONTACTID values from Calls not found in Contacts': (~unique_calls_contactids.isin(contacts_ids)).sum(),

    'Calls rows with CONTACTID': calls_with_contactid.shape[0],
    'Calls rows without CONTACTID': calls_check['CONTACTID_clean'].isna().sum(),

    'Calls rows with CONTACTID found in Contacts': calls_with_contactid['CONTACTID_clean'].isin(contacts_ids).sum(),
    'Calls rows with CONTACTID not found in Contacts': (~calls_with_contactid['CONTACTID_clean'].isin(contacts_ids)).sum()
}

contact_match_check_df = pd.DataFrame(
    contact_match_check.items(),
    columns=['Check', 'Result']
)

contact_match_check_df

,Check,Result
0,Unique Id values in Contacts,18548
1,Unique CONTACTID values in Calls,15214
2,Unique CONTACTID values from Calls found in Co...,15214
3,Unique CONTACTID values from Calls not found i...,0
4,Calls rows with CONTACTID,91941
5,Calls rows without CONTACTID,3933
6,Calls rows with CONTACTID found in Contacts,91941
7,Calls rows with CONTACTID not found in Contacts,0


## `Calls` Data Quality Summary

Based on the audit of the `Calls` table, the main findings are:

- The table contains 95,874 records and 11 columns.
- No full duplicates or duplicates by `Id` were found.
- No missing values were found in key fields `Id`, `Call Start Time`, `Call Owner Name`, `Call Type` and `Call Status`.
- `Call Start Time` can be correctly converted to datetime format.
- `Call Duration (in seconds)` contains 83 missing values/conversion errors. These rows cannot be used for call duration analysis.
- No negative call duration values were found.
- `Dialled Number` and `Tag` are completely empty and will not be used in further analysis.
- `Outgoing Call Status` and `Scheduled in CRM` contain 8,999 missing values. These fields may only be used for additional analysis if needed.
- `CONTACTID` contains 3,933 missing values. These calls can be used for overall sales activity analysis, but they cannot be linked to contacts.
- An additional relationship check showed that all 91,941 rows with a filled `CONTACTID` are correctly found in the `Contacts` table after identifiers are converted to strings.
- Therefore, the `Calls` table can be linked to the `Contacts` table through `CONTACTID`, but only if CRM identifiers are processed as strings.

Overall, the `Calls` table is suitable for sales activity analysis and can be linked to the `Contacts` table.

**Main limitations:** fully empty `Dialled Number` and `Tag` fields, as well as 3,933 calls without `CONTACTID`, which cannot be used in table relationship analysis.

### Preliminary Cleaning Strategy for `Calls`

At the cleaning stage, the following rules will be applied to the `Calls` table:

- Convert `Id` and `CONTACTID` to string type, because they are technical CRM identifiers rather than numerical metrics.
- Use the cleaned `CONTACTID` field to link calls to the `Contacts` table.
- Convert `Call Start Time` to `datetime` format.
- Convert `Call Duration (in seconds)` to numerical type.
- Keep rows with missing or invalid `Call Duration (in seconds)` in the table, but exclude them from call duration analysis.
- Exclude `Dialled Number` and `Tag` from further analysis because they are completely empty.
- Keep rows without `CONTACTID` for overall call analysis, but exclude them from links with `Contacts` and `Deals`.
- Keep `Outgoing Call Status` and `Scheduled in CRM` as additional fields. Use them only if a separate analysis of outbound or scheduled calls is required.
- Keep `Call Type`, `Call Status` and `Call Owner Name` as categorical fields.
- If different spellings of the same category are found, standardize them.
- If categorical fields contain missing values, replace them with `Unknown` to avoid removing full rows.

# Data Quality Audit: `Deals`

As part of the `Deals` audit, I check:

- missing values in key fields;
- correctness of CRM identifiers;
- duplicates;
- correctness of date, SLA and financial field conversion;
- relationship between `Stage` and `Lost Reason`;
- completeness of `Product` and `Education Type` for paid deals;
- relationship between `Stage` and financial fields;
- date logic;
- financial field logic;
- the possibility of linking `Deals` with `Contacts`, `Calls` and `Spend`.

In [56]:
deals_check = deals.copy()

# Clean identifiers to correctly check keys and table relationships

deals_check['Id_clean'] = clean_id_column(deals_check['Id'])
deals_check['Contact Name_clean'] = clean_id_column(deals_check['Contact Name'])

# Check whether dates can be converted to the expected data type

deals_check['Created Time_check'] = pd.to_datetime(
    deals_check['Created Time'],
    errors='coerce',
    dayfirst=True
)

deals_check['Closing Date_check'] = pd.to_datetime(
    deals_check['Closing Date'],
    errors='coerce',
    dayfirst=True
)

# Check whether financial fields can be converted to numerical type

deals_check['Initial Amount Paid_check'] = pd.to_numeric(
    deals_check['Initial Amount Paid'],
    errors='coerce'
)

deals_check['Offer Total Amount_check'] = pd.to_numeric(
    deals_check['Offer Total Amount'],
    errors='coerce'
)

# Check whether SLA can be converted to time format

deals_check['SLA_check'] = pd.to_timedelta(
    deals_check['SLA'].astype(str),
    errors='coerce'
)

deals_checks = {
    'Records': deals.shape[0],
    'Columns': deals.shape[1],
    'Full duplicates': deals.duplicated().sum(),

    'Missing Id': deals_check['Id_clean'].isna().sum(),
    'Duplicate Id values': deals_check['Id_clean'].dropna().duplicated().sum(),

    'Missing Deal Owner Name': deals_check['Deal Owner Name'].isna().sum(),

    'Missing Stage': deals_check['Stage'].isna().sum(),
    'Missing Quality': deals_check['Quality'].isna().sum(),
    'Missing Lost Reason': deals_check['Lost Reason'].isna().sum(),

    'Missing Source': deals_check['Source'].isna().sum(),
    'Missing Campaign': deals_check['Campaign'].isna().sum(),
    'Missing Page': deals_check['Page'].isna().sum(),
    'Missing Content': deals_check['Content'].isna().sum(),
    'Missing Term': deals_check['Term'].isna().sum(),

    'Missing Product': deals_check['Product'].isna().sum(),
    'Missing Education Type': deals_check['Education Type'].isna().sum(),
    'Missing Payment Type': deals_check['Payment Type'].isna().sum(),

    'Missing Created Time': deals_check['Created Time'].isna().sum(),
    'Created Time conversion errors': (
        deals_check['Created Time'].notna() &
        deals_check['Created Time_check'].isna()
    ).sum(),

    'Missing Closing Date': deals_check['Closing Date'].isna().sum(),
    'Closing Date conversion errors': (
        deals_check['Closing Date'].notna() &
        deals_check['Closing Date_check'].isna()
    ).sum(),

    'Missing SLA': deals_check['SLA'].isna().sum(),
    'SLA conversion errors': (
        deals_check['SLA'].notna() &
        deals_check['SLA_check'].isna()
    ).sum(),

    'Missing Course duration': deals_check['Course duration'].isna().sum(),
    'Missing Months of study': deals_check['Months of study'].isna().sum(),

    'Missing Initial Amount Paid': deals_check['Initial Amount Paid'].isna().sum(),
    'Initial Amount Paid conversion errors': (
        deals_check['Initial Amount Paid'].notna() &
        deals_check['Initial Amount Paid_check'].isna()
    ).sum(),

    'Missing Offer Total Amount': deals_check['Offer Total Amount'].isna().sum(),
    'Offer Total Amount conversion errors': (
        deals_check['Offer Total Amount'].notna() &
        deals_check['Offer Total Amount_check'].isna()
    ).sum(),

    'Missing Contact Name': deals_check['Contact Name_clean'].isna().sum(),
    'Missing City': deals_check['City'].isna().sum(),
    'Missing Level of Deutsch': deals_check['Level of Deutsch'].isna().sum()
}

deals_checks_df = pd.DataFrame(
    deals_checks.items(),
    columns=['Check', 'Result']
)

deals_checks_df

,Check,Result
0,Records,21595
1,Columns,23
2,Full duplicates,0
3,Missing Id,2
4,Duplicate Id values,0
5,Missing Deal Owner Name,31
6,Missing Stage,2
7,Missing Quality,2255
8,Missing Lost Reason,5471
9,Missing Source,2


### Detailed Missing Value Check: `Id`, `Source`, `Stage`, `Created Time`

In [57]:
deals_missing_key_fields = deals_check[
    deals_check['Id_clean'].isna() |
    deals_check['Stage'].isna() |
    deals_check['Source'].isna() |
    deals_check['Created Time'].isna()
]

deals_missing_key_fields

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,Term,Source,Payment Type,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch,Id_clean,Contact Name_clean,Created Time_check,Closing Date_check,Initial Amount Paid_check,Offer Total Amount_check,SLA_check
21593,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaT,NaT,NaN,NaN,NaT
21594,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,#REF!,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaT,NaT,NaN,NaN,NaT


### Missing Value Analysis: Key Finding

Rows with missing key fields are empty/technical rows. Rows without `Id`, `Stage`, `Source` and `Created Time` should be excluded at the cleaning stage.

### Detailed Conversion Error Check: `Initial Amount Paid`, `Offer Total Amount`

In [58]:
invalid_initial_amount = deals_check[
    deals_check['Initial Amount Paid'].notna() &
    deals_check['Initial Amount Paid_check'].isna()
]

invalid_initial_amount['Initial Amount Paid'].value_counts(dropna=False)

Initial Amount Paid
€ 3.500,00    16
Name: count, dtype: int64

In [59]:
invalid_offer_amount = deals_check[
    deals_check['Offer Total Amount'].notna() &
    deals_check['Offer Total Amount_check'].isna()
]

invalid_offer_amount['Offer Total Amount'].value_counts(dropna=False)

Offer Total Amount
€ 2.900,00    20
€ 11398,00     1
Name: count, dtype: int64

### Financial Field Check

The financial fields `Initial Amount Paid` and `Offer Total Amount` contain values that cannot be converted to numeric format using the standard `pd.to_numeric` function.

Additional checks showed that the conversion issues are caused by currency symbols (`€`, `$`) and different amount formats. Examples include `€ 3.500,00`, `€ 2.900,00`, `$11,111.00`.

These values are not incorrect from a business perspective, but they require normalization before conversion to numerical type.

### Checking the Relationship Between `Stage` and Financial Fields

According to the dataset annotation, `Payment Done` means that the deal has been paid and money has been received.

I check which deal stages have non-empty and non-zero values in the financial fields:

- `Initial Amount Paid`;
- `Offer Total Amount`.

This helps define which amounts can be used for revenue and unit economics analysis.

In [60]:
deals_check['Initial Amount Paid_clean'] = clean_money_column(
    deals_check['Initial Amount Paid']
)

deals_check['Offer Total Amount_clean'] = clean_money_column(
    deals_check['Offer Total Amount']
)

stage_amount_simple = (
    deals_check
    .groupby('Stage', dropna=False)
    .agg(
        Deals_Count=('Id_clean', 'count'),
        Initial_Amount_Paid_filled=('Initial Amount Paid_clean', lambda x: (x > 0).sum()),
        Offer_Total_Amount_filled=('Offer Total Amount_clean', lambda x: (x > 0).sum())
    )
    .reset_index()
    .sort_values('Deals_Count', ascending=False)
)

stage_amount_simple

,Stage,Deals_Count,Initial_Amount_Paid_filled,Offer_Total_Amount_filled
2,Lost,15743,1752,1794
0,Call Delayed,2248,347,351
10,Registered on Webinar,2072,1,1
7,Payment Done,858,840,840
12,Waiting For Payment,325,324,324
8,Qualificated,128,17,18
9,Registered on Offline Day,100,1,1
5,Need to Call - Sales,33,0,0
3,Need To Call,31,0,0
11,Test Sent,25,6,6


### `Stage` and Financial Fields: Key Findings

The check showed that non-zero values of `Initial Amount Paid` and `Offer Total Amount` are not limited to deals with `Stage = Payment Done`. Therefore, financial fields cannot be used as the only indicator of an actual payment.

According to the dataset description, `Payment Done` means that money has been received. For this reason, the main rule for identifying a paid deal is `Stage == Payment Done`.

`Initial Amount Paid` will be used to calculate actual revenue only for deals with `Stage = Payment Done`.

If a deal has `Stage = Payment Done` but `Initial Amount Paid` is missing, the deal will be included in payment conversion but excluded from revenue and average check calculations because the payment amount is unknown.

`Initial Amount Paid` values for stages other than `Payment Done` will not be interpreted as actual received revenue. They may reflect CRM process specifics, historical stage changes or incomplete data.

Low values in financial fields, such as 0, 1 or 9, are not removed automatically because according to the dataset description they may represent demo access or symbolic payments.

### Checking the Relationship Between `Stage` and `Lost Reason`

`Lost Reason` contains the reason why a deal was lost.  
Based on the business logic, this field should be filled for deals with `Stage = Lost`.

I check:
- how many deals are in the `Lost` stage;
- how many `Lost` deals do not have a lost reason;
- whether there are deals not in `Lost` stage but with a filled `Lost Reason`.

In [61]:
stage_lost_reason_pivot = pd.pivot_table(
    deals_check,
    index='Stage',
    columns='Lost Reason',
    values='Id_clean',
    aggfunc='count',
    fill_value=0,
    dropna=False
)

stage_lost_reason_pivot

Lost Reason,Changed Decision,Conditions are not suitable,Considering a different direction in IT,Didn't leave an application,Does not know how to use a computer,Does not speak English,Doesn't Answer,Duplicate,Expensive,Gutstein refusal,Inadequate,Invalid number,Next stream,Non target,Not for myself,Refugee,Stopped Answering,The contract did not fit,Thought for free,Went to Rivals,needs time to think,NaN
Stage,,,,,,,,,,,,,,,,,,,,,,
Call Delayed,12,3,0,2,0,0,38,5,5,4,1,17,112,15,0,0,15,0,0,1,27,1991
Free Education,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
Lost,2122,524,148,131,49,138,4074,1746,614,163,174,1460,131,1736,145,1,1556,21,110,47,606,47
Need To Call,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,31
Need a consultation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,23
Need to Call - Sales,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,31
New Lead,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6
Payment Done,7,4,0,0,1,0,15,2,6,3,0,3,40,4,0,0,15,0,0,0,15,743
Qualificated,2,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,125


### `Stage` and `Lost Reason`: Key Findings

The check showed that `Lost Reason` is not used strictly only for deals in the `Lost` stage.

`Lost Reason` values appear not only for lost deals but also for other stages, including `Payment Done`. This may mean that the field keeps historical information about a client's previous objection or that it is not always cleared when the deal stage changes.

Therefore, `Lost Reason` should not be used as an independent indicator of the deal status. The main field for the current deal status remains `Stage`.

Missing values in `Lost Reason` are not filled automatically and are not treated as an error for deals that are not in the `Lost` stage.

In further analysis, `Lost Reason` will be used only as an additional field for analyzing reasons for loss and barriers together with `Stage`.

### Detailed Analysis of `Product` and `Education Type`

There is a substantial number of missing values in `Product` (18,003) and `Education Type` (18,295).

According to the dataset annotation, `Payment Done` means that the deal has been paid and money has been received.

Therefore, `Product` and `Education Type` are critical not for all deals, but primarily for paid deals.  
For other stages, missing values in these fields may be a normal CRM feature: the client may not have reached the product or study format selection step.

In [62]:
paid_deals = deals_check[
    deals_check['Stage'] == 'Payment Done'
].copy()

product_education_checks = {
    'Payment Done deals': paid_deals.shape[0],

    'Payment Done without Product': paid_deals['Product'].isna().sum(),
    'Payment Done with Product': paid_deals['Product'].notna().sum(),

    'Payment Done without Education Type': paid_deals['Education Type'].isna().sum(),
    'Payment Done with Education Type': paid_deals['Education Type'].notna().sum(),

    'Payment Done without both Product and Education Type': (
        paid_deals['Product'].isna() &
        paid_deals['Education Type'].isna()
    ).sum()
}

product_education_checks_df = pd.DataFrame(
    product_education_checks.items(),
    columns=['Check', 'Result']
)

product_education_checks_df

,Check,Result
0,Payment Done deals,858
1,Payment Done without Product,17
2,Payment Done with Product,841
3,Payment Done without Education Type,25
4,Payment Done with Education Type,833
5,Payment Done without both Product and Educatio...,17


### `Product` and `Education Type`: Key Findings

`Product` and `Education Type` are critical primarily for deals with `Stage = Payment Done`, because this status confirms that money has been received from the client.

The table contains 858 paid deals (`Payment Done`).

Among them:
- 17 deals have no filled `Product`;
- 25 deals have no filled `Education Type`;
- 17 deals have neither `Product` nor `Education Type`.

These missing values are important for product analysis, but they are not a reason to remove the rows because the deals have already generated revenue.

At the cleaning stage, missing `Product` and `Education Type` values for `Payment Done` deals will be filled with the technical value `Unknown`. This keeps paid deals in revenue and conversion analysis while identifying them as deals without a known product or study format.

### Checking Date Logic in `Deals`

I check the relationship between `Created Time` and `Closing Date`.

From a business logic perspective, a deal closing date should not be earlier than the deal creation date.  
If such rows exist, they cannot be used for deal duration analysis without additional handling.

At this stage, the data is not cleaned yet. I only identify potential logical errors.

In [63]:
# Check Deals date logic using calendar dates only

deals_check['Created Date_check'] = deals_check['Created Time_check'].dt.normalize()
deals_check['Closing Date_only_check'] = deals_check['Closing Date_check'].dt.normalize()

deals_invalid_dates = deals_check[
    deals_check['Created Date_check'].notna() &
    deals_check['Closing Date_only_check'].notna() &
    (deals_check['Closing Date_only_check'] < deals_check['Created Date_check'])
]

date_logic_checks = {
    'Rows with filled Created Time and Closing Date': (
        deals_check['Created Date_check'].notna() &
        deals_check['Closing Date_only_check'].notna()
    ).sum(),

    'Rows where Closing Date is earlier than Created Time by calendar date': deals_invalid_dates.shape[0]
}

date_logic_checks_df = pd.DataFrame(
    date_logic_checks.items(),
    columns=['Check', 'Result']
)

date_logic_checks_df

,Check,Result
0,Rows with filled Created Time and Closing Date,14645
1,Rows where Closing Date is earlier than Create...,44


In [64]:
deals_invalid_dates[
    [
        'Id_clean',
        'Stage',
        'Created Time',
        'Closing Date',
        'Created Date_check',
        'Closing Date_only_check',
        'Source',
        'Campaign',
        'Deal Owner Name'
    ]
].head(20)

,Id_clean,Stage,Created Time,Closing Date,Created Date_check,Closing Date_only_check,Source,Campaign,Deal Owner Name
454,5805028000055502890,Lost,16.06.2024 00:06,11.06.2024,2024-06-16,2024-06-11,Facebook Ads,24.09.23retargeting_DE,Quincy Vincent
2083,5805028000051847114,Lost,25.05.2024 21:29,22.05.2024,2024-05-25,2024-05-22,Tiktok Ads,22.05.2024wide_DE,Quincy Vincent
2787,5805028000049539444,Lost,12.05.2024 11:19,07.05.2024,2024-05-12,2024-05-07,SMM,NaN,Julia Nelson
3019,5805028000048886321,Payment Done,08.05.2024 15:31,07.05.2024,2024-05-08,2024-05-07,Organic,NaN,Oliver Taylor
3022,5805028000048883316,Lost,08.05.2024 14:48,17.04.2024,2024-05-08,2024-04-17,Facebook Ads,03.07.23women,Ulysses Adams
3031,5805028000048886160,Payment Done,08.05.2024 12:54,07.05.2024,2024-05-08,2024-05-07,Organic,NaN,Oliver Taylor
3691,5805028000047482214,Lost,30.04.2024 15:16,23.04.2024,2024-04-30,2024-04-23,Tiktok Ads,12.07.2023wide_DE,Paula Underwood
4107,5805028000046234250,Lost,24.04.2024 17:30,17.04.2024,2024-04-24,2024-04-17,Organic,NaN,Rachel White
4169,5805028000045961466,Lost,23.04.2024 21:44,18.04.2024,2024-04-23,2024-04-18,Tiktok Ads,12.07.2023wide_DE,Paula Underwood
4434,5805028000045314301,Lost,21.04.2024 08:57,18.04.2024,2024-04-21,2024-04-18,Facebook Ads,02.07.23wide_DE,Quincy Vincent


### Date Logic: Key Findings

After comparing calendar dates in the `Deals` table, 44 rows were found where `Closing Date` is earlier than `Created Time`.

These rows represent a small share of the data and are not critical for the overall analysis of deals, sources, stages and payments.

At the cleaning stage, these rows will not be removed. A date logic quality flag will be created for them, for example `invalid_deal_dates = True`.

Rows with this flag can remain in the overall analysis, but should be excluded from deal duration and time-to-close analysis.

### Checking Financial Field Logic in `Deals`

I check the relationship between `Initial Amount Paid` and `Offer Total Amount`.

From a business logic perspective, a client's first payment should not exceed the total offer amount.  
If such rows exist, they need to be checked separately before financial analysis.

The check uses cleaned numerical versions of the fields:
- `Initial Amount Paid_clean`;
- `Offer Total Amount_clean`.

In [65]:
deals_invalid_amounts = deals_check[
    deals_check['Initial Amount Paid_clean'].notna() &
    deals_check['Offer Total Amount_clean'].notna() &
    (deals_check['Initial Amount Paid_clean'] > deals_check['Offer Total Amount_clean'])
]

amount_logic_checks = {
    'Rows with filled Initial Amount Paid and Offer Total Amount': (
        deals_check['Initial Amount Paid_clean'].notna() &
        deals_check['Offer Total Amount_clean'].notna()
    ).sum(),

    'Rows where Initial Amount Paid > Offer Total Amount': deals_invalid_amounts.shape[0]
}

amount_logic_checks_df = pd.DataFrame(
    amount_logic_checks.items(),
    columns=['Check', 'Result']
)

amount_logic_checks_df

,Check,Result
0,Rows with filled Initial Amount Paid and Offer...,4159
1,Rows where Initial Amount Paid > Offer Total A...,58


In [66]:
deals_invalid_amounts[
    [
        'Id_clean',
        'Stage',
#        'Initial Amount Paid',
        'Initial Amount Paid_clean',
#        'Offer Total Amount',
        'Offer Total Amount_clean',
        'Payment Type',
        'Product',
        'Source'
    ]
].head(20)

,Id_clean,Stage,Initial Amount Paid_clean,Offer Total Amount_clean,Payment Type,Product,Source
1279,5805028000053717506,Call Delayed,"3,000.00","2,900.00",NaN,Web Developer,SMM
1393,5805028000053561185,Lost,"11,500.00","11,000.00",NaN,UX/UI Design,Telegram posts
1409,5805028000053462041,Lost,"11,500.00","11,000.00",NaN,UX/UI Design,Google Ads
1440,5805028000053242748,Waiting For Payment,"11,500.00","11,000.00",NaN,UX/UI Design,SMM
1452,5805028000053242571,Lost,"11,500.00","11,000.00",NaN,UX/UI Design,Telegram posts
1484,5805028000053252538,Payment Done,"3,000.00","2,900.00",One Payment,Web Developer,Facebook Ads
1703,5805028000052969001,Waiting For Payment,"11,500.00","11,000.00",Recurring Payments,UX/UI Design,SMM
1967,5805028000052197344,Lost,"3,000.00","2,500.00",One Payment,Web Developer,Google Ads
2060,5805028000051885327,Waiting For Payment,"11,500.00","11,000.00",NaN,UX/UI Design,Tiktok Ads
2324,5805028000050859250,Lost,"11,500.00","11,000.00",NaN,UX/UI Design,Facebook Ads


### Financial Field Logic: Key Findings

The `Deals` table contains 58 rows where `Initial Amount Paid` is greater than `Offer Total Amount`.

At first glance, this violates the expected business logic: a client's first payment should not exceed the total offer amount.

However, additional row-level inspection shows that many differences are relatively small and may be related to CRM/accounting specifics: changes in the final offer amount, recalculation, partial refunds or lack of detailed treasury/payment operations in the dataset.

Because the dataset does not include a separate payments, refunds or financial adjustments table, these values will not be corrected manually.

At the cleaning stage, a data quality flag `initial_amount_gt_offer = True` will be created for such rows.

These deals can remain in the overall analysis of payments and conversion, but they should be treated separately when analyzing the relationship between first payment and total offer amount.

### Checking Relationships Between Tables

I check key fields that can potentially link the project tables:

- `Deals.Deal Owner Name` ↔ `Calls.Call Owner Name`;
- `Deals.Source` ↔ `Spend.Source`;
- `Deals.Campaign` ↔ `Spend.Campaign`;
- `Deals.Term` ↔ `Spend.AdGroup`;
- `Deals.Content` ↔ `Spend.Ad`;
- `Deals.Contact Name` → `Contacts.Id`.

The goal is to understand which relationships can be used for further analysis and which require careful interpretation or additional handling.

In [67]:
# Check relationship between Deals.Deal Owner Name and Calls.Call Owner Name

deals_check['Deal Owner Name_clean'] = (
    deals_check['Deal Owner Name']
    .astype('string')
    .str.strip()
)

calls_check['Call Owner Name_clean'] = (
    calls_check['Call Owner Name']
    .astype('string')
    .str.strip()
)

deal_owners = deals_check['Deal Owner Name_clean'].dropna().drop_duplicates()
call_owners = calls_check['Call Owner Name_clean'].dropna().drop_duplicates()

deal_owners_set = set(deal_owners)
call_owners_set = set(call_owners)

owners_match_check = {
    'Unique managers in Deals': deal_owners.nunique(),
    'Unique managers in Calls': call_owners.nunique(),

    'Managers from Deals found in Calls': deal_owners.isin(call_owners_set).sum(),
    'Managers from Deals not found in Calls': (~deal_owners.isin(call_owners_set)).sum(),

    'Managers from Calls found in Deals': call_owners.isin(deal_owners_set).sum(),
    'Managers from Calls not found in Deals': (~call_owners.isin(deal_owners_set)).sum()
}

owners_match_check_df = pd.DataFrame(
    owners_match_check.items(),
    columns=['Check', 'Result']
)

owners_match_check_df

,Check,Result
0,Unique managers in Deals,27
1,Unique managers in Calls,33
2,Managers from Deals found in Calls,27
3,Managers from Deals not found in Calls,0
4,Managers from Calls found in Deals,27
5,Managers from Calls not found in Deals,6


### `Deal Owner Name` and `Call Owner Name`: Key Findings

The check showed that all managers from the `Deals` table are found in the `Calls` table.

The `Deals` table contains 27 unique managers, and all of them are present among the managers who made calls in the `Calls` table. This means that the manager reference values are consistent across the two tables and can be used for further sales activity analysis.

At the same time, the `Calls` table contains 6 managers who are not present in `Deals`. This is not considered a data error, because a manager may make calls without being the owner of a deal in the analyzed table.

The relationship between `Deal Owner Name` and `Call Owner Name` can be used for aggregated manager-level analysis, but not as a strict key for joining specific deals and calls.

In [68]:
# Check advertising relationships between Deals and Spend

deals_check['Source_clean'] = clean_text_column(deals_check['Source'])
deals_check['Campaign_clean'] = clean_text_column(deals_check['Campaign'])
deals_check['Term_clean'] = clean_text_column(deals_check['Term'])
deals_check['Content_clean'] = clean_text_column(deals_check['Content'])

spend_check['Source_clean'] = clean_text_column(spend_check['Source'])
spend_check['Campaign_clean'] = clean_text_column(spend_check['Campaign'])
spend_check['AdGroup_clean'] = clean_text_column(spend_check['AdGroup'])
spend_check['Ad_clean'] = clean_text_column(spend_check['Ad'])


def check_field_match(left_df, left_col, right_df, right_col, left_name, right_name):
    left_values = left_df[left_col].dropna().drop_duplicates()
    right_values = right_df[right_col].dropna().drop_duplicates()

    right_set = set(right_values)
    left_set = set(left_values)

    return {
        f'Unique {left_name}': left_values.nunique(),
        f'Unique {right_name}': right_values.nunique(),
        f'{left_name} found in {right_name}': left_values.isin(right_set).sum(),
        f'{left_name} not found in {right_name}': (~left_values.isin(right_set)).sum(),
        f'{right_name} found in {left_name}': right_values.isin(left_set).sum(),
        f'{right_name} not found in {left_name}': (~right_values.isin(left_set)).sum(),
        f'Deals rows with {left_name}': left_df[left_col].notna().sum(),
        f'Deals rows without {left_name}': left_df[left_col].isna().sum(),
        f'Spend rows with {right_name}': right_df[right_col].notna().sum(),
        f'Spend rows without {right_name}': right_df[right_col].isna().sum()
    }


ads_match_checks = {}

ads_match_checks.update(
    check_field_match(
        deals_check, 'Source_clean',
        spend_check, 'Source_clean',
        'Deals.Source', 'Spend.Source'
    )
)

ads_match_checks.update(
    check_field_match(
        deals_check, 'Campaign_clean',
        spend_check, 'Campaign_clean',
        'Deals.Campaign', 'Spend.Campaign'
    )
)

ads_match_checks.update(
    check_field_match(
        deals_check, 'Term_clean',
        spend_check, 'AdGroup_clean',
        'Deals.Term', 'Spend.AdGroup'
    )
)

ads_match_checks.update(
    check_field_match(
        deals_check, 'Content_clean',
        spend_check, 'Ad_clean',
        'Deals.Content', 'Spend.Ad'
    )
)

ads_match_checks_df = pd.DataFrame(
    ads_match_checks.items(),
    columns=['Check', 'Result']
)

ads_match_checks_df

,Check,Result
0,Unique Deals.Source,13
1,Unique Spend.Source,14
2,Deals.Source found in Spend.Source,13
3,Deals.Source not found in Spend.Source,0
4,Spend.Source found in Deals.Source,13
5,Spend.Source not found in Deals.Source,1
6,Deals rows with Deals.Source,21593
7,Deals rows without Deals.Source,2
8,Spend rows with Spend.Source,20779
9,Spend rows without Spend.Source,0


In [69]:
# Examples of Deals.Campaign values not found in Spend.Campaign
campaigns_deals_not_in_spend = (
    deals_check['Campaign_clean']
    .dropna()
    .drop_duplicates()
)

campaigns_deals_not_in_spend = campaigns_deals_not_in_spend[
    ~campaigns_deals_not_in_spend.isin(set(spend_check['Campaign_clean'].dropna()))
]

campaigns_deals_not_in_spend.head(30)

2                              engwien_AT
9                               1406start
19                              1006start
24                             germany_DE
35                            webinar1906
45                            germania_DE
78                                blog_DE
113                       Jobs_germany_DE
136                          2005_Lost_DE
139                                 uk_DE
194                              Akademia
346                           BloggerIvan
373                              Genie_DE
456                               Live_DE
471                               1706_DE
606     performancemax_digitalmarkt_ru_DE
655                               5555_DE
843                             ASA_de_DE
865                             2905start
1002                          webinar1604
1041                       bloggerfrai_DE
1062                         bloggerdr_DE
1085                           Trigger_DE
1156                         Blogg

In [70]:
# Examples of Deals.Term values not found in Spend.AdGroup
terms_deals_not_in_spend = (
    deals_check['Term_clean']
    .dropna()
    .drop_duplicates()
)

terms_deals_not_in_spend = terms_deals_not_in_spend[
    ~terms_deals_not_in_spend.isin(set(spend_check['AdGroup_clean'].dropna()))
]

terms_deals_not_in_spend.head(30)

2                 21_06_2024
7              it career hub
24                21_05_2024
35                invitation
45                19_06_2024
91                30_05_2024
111                      ich
113               10_04_2024
137        it%20career%20hub
139               18_06_2024
166              invitation\
230                        _
346               07_06_2024
373               14_06_2024
396               13_06_2024
456               03_06_2024
655               12_06_2024
843               28_05_2024
894                   it hub
932              itcareerhub
1041              31_05_2024
1062              05_06_2024
1079          it career hub_
1156              15_05_2024
1210              29_03_2024
1227    lost_does_not_answer
1264         айти карьер хаб
1292              23_05_2024
1358              13_01_2024
1367            1_day_before
Name: Term_clean, dtype: string

In [71]:
# Examples of Deals.Content values not found in Spend.Ad
content_deals_not_in_spend = (
    deals_check['Content_clean']
    .dropna()
    .drop_duplicates()
)

content_deals_not_in_spend = content_deals_not_in_spend[
    ~content_deals_not_in_spend.isin(set(spend_check['Ad_clean'].dropna()))
]

content_deals_not_in_spend.head(30)

2                                          b1-at
4                                        website
7        152789402780_{region_name}_695563281558
17                               _{region_name}_
24                                            b9
27                                  search_terms
52                                      Audience
75       152789402780_{region_name}_668024583824
104                               bloggersjune17
139                                           b0
153      151836595805_{region_name}_699672039100
224      151836595805_{region_name}_699672039103
236      151836595805_{region_name}_699672039109
492      151836595805_{region_name}_699672039106
531      151836595805_{region_name}_673801336999
1085                                        ntc1
1241    Com_august_{region_name}_bloggersvideo10
1450             bloggersvideo10com#rec717852307
1538                                       b2-at
2388                                       b2-pl
2528     15183659580

### Advertising Field Relationships Between `Deals` and `Spend`: Key Findings

The advertising field check showed that `Deals` and `Spend` can be matched, but the reliability of the relationship depends on the level of detail.

The most stable relationship is `Source`:

- 13 unique sources were found in `Deals`;
- all 13 sources from `Deals` were found in `Spend`;
- `Spend` contains 1 source that is not present in `Deals`.

This means that the advertising channel level (`Source`) can be used as the main level for marketing efficiency analysis: spend, leads, deals, paid deals and conversion.

The `Campaign` relationship is partial:

- 154 unique campaigns were found in `Deals`;
- 51 unique campaigns were found in `Spend`;
- 45 campaigns match;
- 109 campaigns from `Deals` are not found in `Spend`;
- 6 campaigns from `Spend` are not found in `Deals`.

This means that campaign-level analysis is possible only for part of the data. `Campaign` can be used as an additional level of detail, but not as the only key for linking marketing spend and deals.

The relationship between `Deals.Term` and `Spend.AdGroup` is weak:

- 220 unique `Term` values were found in `Deals`;
- 24 unique `AdGroup` values were found in `Spend`;
- only 19 values match.

This indicates that the ad group level is poorly aligned between the CRM and advertising spend export. `Term` can be used only as a supporting field, not as a reliable key for marketing efficiency analysis.

The relationship between `Deals.Content` and `Spend.Ad` is better but still incomplete:

- 187 unique `Content` values were found in `Deals`;
- 176 unique `Ad` values were found in `Spend`;
- 138 values match;
- some ads are present only in one of the two tables.

This means that ad/creative-level analysis is possible only partially and requires careful interpretation.

In further analysis, marketing efficiency should mainly be analyzed at the `Source` level because it is the most complete and stable relationship between `Deals` and `Spend`.

`Campaign`, `Term`, `Content`, `AdGroup` and `Ad` can be used as additional details only where values are correctly matched between the tables.

In [72]:
# Check relationship between Deals.Contact Name and Contacts.Id

contacts_ids = set(contacts_check['Id_clean'].dropna())

deals_with_contact = deals_check[
    deals_check['Contact Name_clean'].notna()
].copy()

unique_deals_contact_ids = deals_with_contact['Contact Name_clean'].drop_duplicates()

deals_contacts_match_check = {
    'Unique Id values in Contacts': contacts_check['Id_clean'].nunique(),
    'Unique Contact Name values in Deals': unique_deals_contact_ids.nunique(),

    'Unique Contact Name values from Deals found in Contacts': unique_deals_contact_ids.isin(contacts_ids).sum(),
    'Unique Contact Name values from Deals not found in Contacts': (~unique_deals_contact_ids.isin(contacts_ids)).sum(),

    'Deals rows with Contact Name': deals_with_contact.shape[0],
    'Deals rows without Contact Name': deals_check['Contact Name_clean'].isna().sum(),

    'Deals rows with Contact Name found in Contacts': deals_with_contact['Contact Name_clean'].isin(contacts_ids).sum(),
    'Deals rows with Contact Name not found in Contacts': (~deals_with_contact['Contact Name_clean'].isin(contacts_ids)).sum()
}

deals_contacts_match_check_df = pd.DataFrame(
    deals_contacts_match_check.items(),
    columns=['Check', 'Result']
)

deals_contacts_match_check_df

,Check,Result
0,Unique Id values in Contacts,18548
1,Unique Contact Name values in Deals,18089
2,Unique Contact Name values from Deals found in...,18088
3,Unique Contact Name values from Deals not foun...,1
4,Deals rows with Contact Name,21532
5,Deals rows without Contact Name,63
6,Deals rows with Contact Name found in Contacts,21531
7,Deals rows with Contact Name not found in Cont...,1


In [73]:
deals_contacts_not_found = deals_with_contact[
    ~deals_with_contact['Contact Name_clean'].isin(contacts_ids)
]

deals_contacts_not_found[
    ['Id_clean', 'Contact Name', 'Contact Name_clean', 'Stage', 'Created Time', 'Source', 'Campaign']
].head(20)

,Id_clean,Contact Name,Contact Name_clean,Stage,Created Time,Source,Campaign
1840,5805028000052515051,5805028000052408019,5805028000052408019,Lost,30.05.2024 06:50,Organic,NaN


### `Deals.Contact Name` and `Contacts.Id`: Key Findings

The check showed that `Contact Name` in the `Deals` table can be used as a contact identifier for linking deals to the `Contacts` table.

Out of 21,532 `Deals` rows with a filled `Contact Name`, 21,531 rows are correctly found in the `Contacts` table.

Only 1 `Contact Name` value was not found. This is a non-critical discrepancy and does not materially affect the ability to link deals with contacts.

Rows without `Contact Name` or with a `Contact Name` not found in `Contacts` will not be removed from the overall deal analysis, but they cannot be used for relationship analysis between tables.

## `Deals` Data Quality Summary

Based on the audit of the `Deals` table, the main findings are:

- The table contains 21,595 records and is the main table of the project.
- After CRM identifiers are correctly processed as strings, no duplicates by `Id` were found. No full duplicates were found either.
- The table contains 2 rows with missing values in key fields: `Id`, `Stage`, `Source` and `Created Time`. These rows should be excluded at the cleaning stage because they cannot be correctly used in deal analysis.
- `Created Time` and `Closing Date` can be correctly converted to date format. `Created Time` contains date and time, while `Closing Date` contains only the date, so calendar dates should be compared when checking date logic and calculating deal duration.
- After comparing calendar dates, 44 rows were found where `Closing Date` is earlier than `Created Time`. These rows are not critical for the overall analysis, but they cannot be used for deal duration calculation.
- `SLA` can be correctly converted to a time format; no conversion errors were found. Rows with missing `SLA` can remain in the overall analysis, but should be excluded from lead response speed analysis.
- The financial fields `Initial Amount Paid` and `Offer Total Amount` contain different formats: regular numbers, amounts with currency symbols, commas, dots and spaces. These fields require technical normalization before conversion to numerical type.
- `Stage` is the main field for determining the current deal status.
- According to the dataset annotation, `Payment Done` means that money has been received. Therefore, `Stage == Payment Done` is used as the main indicator of an actual payment.
- `Initial Amount Paid` is used as the actual first payment amount only for deals with `Stage = Payment Done`.
- `Offer Total Amount` represents the offer/course value and is not an independent indicator of actual payment.
- Non-zero values in financial fields are not limited to `Payment Done` deals, so the amounts themselves are not used to identify paid deals.
- The table contains 58 rows where `Initial Amount Paid` is greater than `Offer Total Amount`. These rows may be related to recalculations, refunds, offer changes or CRM payment recording specifics. Since the dataset does not contain detailed treasury operations, these values are not corrected manually.
- `Lost Reason` is not used strictly only for `Lost` deals: values also appear in other stages, including `Payment Done`. Therefore, `Lost Reason` cannot be used as an independent indicator of a lost deal and should only be analyzed together with `Stage`.
- `Product` and `Education Type` are critical primarily for `Payment Done` deals. Among paid deals, a small number of missing values was found: 17 deals without `Product`, 25 deals without `Education Type`, and 17 deals without both fields.
- The relationship `Deals.Contact Name → Contacts.Id` is almost fully valid: out of 21,532 rows with a filled `Contact Name`, only 1 value is not found in `Contacts`.
- All managers from `Deals.Deal Owner Name` were found in `Calls.Call Owner Name`, so manager reference values are consistent across these tables.
- Advertising fields in `Deals` and `Spend` are best matched at the `Source` level. Relationships by `Campaign`, `Term / AdGroup` and `Content / Ad` are partial and should be used only as additional detail.

Overall, the `Deals` table is suitable for further analysis, but it requires careful cleaning and additional technical flags for correct use of dates, amounts, payments, products and relationships between tables.

### Preliminary Cleaning Strategy for `Deals`

At the cleaning stage, the following rules will be applied to the `Deals` table:

- Process all CRM identifiers as strings:
  - `Id`;
  - `Contact Name`.
- Use the cleaned `Id` field as the deal identifier.
- Use the cleaned `Contact Name` field to link deals with `Contacts`.
- Exclude rows without key fields `Id`, `Stage`, `Source` and `Created Time`, because they cannot be correctly used in deal analysis.
- Convert `Created Time` to `datetime`.
- Convert `Closing Date` to date format.
- Use calendar dates without time when checking date logic and calculating deal duration.
- For rows where `Closing Date` is earlier than `Created Time`, create a flag `invalid_deal_dates = True`.
- Keep rows with `invalid_deal_dates = True` in the overall analysis, but exclude them from deal duration calculation.
- Convert `SLA` to time / timedelta format.
- Keep rows with missing `SLA` in the table, but exclude them from lead response speed analysis.
- Clean the financial fields `Initial Amount Paid` and `Offer Total Amount`:
  - remove currency symbols (`€`, `$`);
  - remove spaces;
  - standardize thousand and decimal separators;
  - convert values to numerical type.
- Create a paid deal flag `is_paid` using the rule `Stage == Payment Done`.
- Use `Initial Amount Paid` to calculate actual revenue only for deals with `Stage = Payment Done`.
- If `Stage = Payment Done` but `Initial Amount Paid` is missing, keep the deal in payment conversion analysis but exclude it from revenue and average check calculations.
- Do not treat `Initial Amount Paid` values in stages other than `Payment Done` as actual received revenue.
- Do not automatically remove low financial values such as 0, 1 or 9, because according to the dataset description they may represent demo access or symbolic payments.
- For rows where `Initial Amount Paid > Offer Total Amount`, create a flag `initial_amount_gt_offer = True`.
- Do not automatically remove or manually correct these rows because the dataset does not include refunds, recalculations or treasury operations.
- Do not automatically fill `Lost Reason`.
- Use `Lost Reason` only as a supporting field together with `Stage`.
- If `Lost Reason` is filled for a stage other than `Lost`, keep the value.
- For `Payment Done` deals with missing `Product` or `Education Type`, fill missing values with `Unknown`.
- Keep `Payment Done` deals with `Product = Unknown` or `Education Type = Unknown` in payment and conversion analysis, but treat them as a separate category in product analysis.
- Do not fill missing `Product` and `Education Type` values for stages other than `Payment Done`.
- Build the main relationship with marketing spend at the `Source` level.
- Use `Campaign`, `Term` and `Content` as additional details only where they correctly match values in the `Spend` table.